# Hybrid NIDS: Fusing Suricata + Supervised MLP + Unsupervised Autoencoder

Combines all three detection layers into one alert pipeline with a unified severity score:

| Source | Detects | Confidence when it fires alone |
|---|---|---|
| Suricata (signatures) | Known attack patterns in packet/rule form | High |
| Supervised MLP | Known attack *types* from flow features | High |
| Autoencoder (unsupervised) | Anything structurally abnormal, including unseen attack types | Medium (needs corroboration) |

**Fusion logic:** each flow gets a score from each available source; scores combine into one severity tier. Agreement across sources escalates severity sharply — that's the strongest possible signal of a real incident, versus a single source firing on its own, which could be a false positive or genuinely-novel-but-uncertain traffic.

## Step 1: Load All Three Artifact Sets

In [7]:
import os
import json
import numpy as np
import pandas as pd
import joblib
from tensorflow import keras

MODELS_DIR = "models"

# Supervised MLP artifacts
mlp_model = keras.models.load_model(os.path.join(MODELS_DIR, "neural_network_ids.h5"))
mlp_scaler = joblib.load(os.path.join(MODELS_DIR, "scaler.pkl"))
label_encoder = joblib.load(os.path.join(MODELS_DIR, "label_encoder.pkl"))
mlp_feature_names = list(getattr(mlp_scaler, "feature_names_in_", []))

# Unsupervised autoencoder artifacts
autoencoder = keras.models.load_model(os.path.join(MODELS_DIR, "nids_autoencoder.h5"))
ae_scaler = joblib.load(os.path.join(MODELS_DIR, "ae_scaler.pkl"))
ae_meta = joblib.load(os.path.join(MODELS_DIR, "ae_threshold.pkl"))
ae_threshold = ae_meta["threshold"]
ae_feature_names = ae_meta["feature_names"]

print("Supervised classes:", list(label_encoder.classes_))
print("Autoencoder threshold:", ae_threshold)


Supervised classes: ['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SSH-Patator', 'Web Attack � Brute Force', 'Web Attack � Sql Injection', 'Web Attack � XSS']
Autoencoder threshold: 0.12445472806275219


## Step 2: Suricata Alert Ingestion
Suricata writes alerts to `eve.json` (one JSON object per line) when `outputs.eve-log` is enabled with the `alert` event type. This reads that log and indexes alerts by flow identifiers (src/dst IP + port + protocol) so they can be joined against the same flow records the ML models score.

Adjust `EVE_JSON_PATH` to wherever your Suricata instance writes it (commonly `/var/log/suricata/eve.json`).

In [8]:
# EVE_JSON_PATH = "/var/log/suricata/eve.json"
EVE_JSON_PATH = "eve.json"

def load_suricata_alerts(path: str) -> pd.DataFrame:
    """Parse Suricata's eve.json, keeping only 'alert' events."""
    if not os.path.exists(path):
        print(f"'{path}' not found - returning an empty alert table. "
              "Point EVE_JSON_PATH at your Suricata eve.json to enable signature correlation.")
        return pd.DataFrame(columns=["src_ip", "dest_ip", "src_port", "dest_port", "proto", "signature", "severity"])

    rows = []
    with open(path) as f:
        for line in f:
            try:
                event = json.loads(line)
            except json.JSONDecodeError:
                continue
            if event.get("event_type") != "alert":
                continue
            alert = event.get("alert", {})
            rows.append({
                "src_ip": event.get("src_ip"),
                "dest_ip": event.get("dest_ip"),
                "src_port": event.get("src_port"),
                "dest_port": event.get("dest_port"),
                "proto": event.get("proto"),
                "signature": alert.get("signature"),
                "severity": alert.get("severity"),  # Suricata: 1=high ... 3=low
            })
    return pd.DataFrame(rows)

suricata_alerts = load_suricata_alerts(EVE_JSON_PATH)
print(f"Loaded {len(suricata_alerts):,} Suricata alerts.")
suricata_alerts.head()


Loaded 39 Suricata alerts.


,src_ip,dest_ip,src_port,dest_port,proto,signature,severity
0,2620:002d:4000:0001:0000:0000:0000:1100,2402:d000:a400:cbae:ba2e:0b63:17d3:ff4a,80.0,55442.0,TCP,SURICATA HTTP unable to match response to request,3
1,146.75.45.91,192.168.1.112,80.0,56474.0,TCP,SURICATA STREAM TIMEWAIT ACK with wrong seq,3
2,146.75.45.91,192.168.1.112,80.0,56518.0,TCP,SURICATA STREAM TIMEWAIT ACK with wrong seq,3
3,146.75.45.91,192.168.1.112,80.0,56542.0,TCP,SURICATA STREAM TIMEWAIT ACK with wrong seq,3
4,146.75.45.91,192.168.1.112,80.0,56576.0,TCP,SURICATA STREAM TIMEWAIT ACK with wrong seq,3


**Matching alerts to flows:** Suricata alerts are per-packet/per-rule events, while your flow records are aggregated (CICFlowMeter-style) — so join on the 5-tuple (`src_ip`, `dest_ip`, `src_port`, `dest_port`, `proto`) if your flow CSV has those columns. If your exported flow features don't retain IPs/ports (common after feature selection), you'll need to keep a side-table mapping flow ID → 5-tuple upstream of feature selection, purely for this join - it never needs to touch the model.

In [9]:
def attach_suricata_flags(flows_df: pd.DataFrame, alerts_df: pd.DataFrame) -> pd.Series:
    """Returns a boolean Series aligned to flows_df.index: True if a Suricata alert
    matches that flow's 5-tuple."""
    required_cols = {"src_ip", "dest_ip", "src_port", "dest_port", "proto"}
    if alerts_df.empty or not required_cols.issubset(flows_df.columns):
        return pd.Series(False, index=flows_df.index)

    alert_keys = set(
        zip(alerts_df["src_ip"], alerts_df["dest_ip"], alerts_df["src_port"],
            alerts_df["dest_port"], alerts_df["proto"])
    )
    flow_keys = list(
        zip(flows_df["src_ip"], flows_df["dest_ip"], flows_df["src_port"],
            flows_df["dest_port"], flows_df["proto"])
    )
    return pd.Series([k in alert_keys for k in flow_keys], index=flows_df.index)

print("attach_suricata_flags() ready.")


attach_suricata_flags() ready.


## Step 3: Score Each Flow With Both Neural Networks

In [10]:
def preprocess_for(df_raw: pd.DataFrame, feature_names: list) -> pd.DataFrame:
    df = df_raw.copy()
    df.columns = df.columns.str.strip()
    for lbl_col in ("Label", "label", "Class", "class"):
        if lbl_col in df.columns:
            df = df.drop(columns=[lbl_col])
    df = df.select_dtypes(include=[np.number])
    if feature_names:
        missing = [c for c in feature_names if c not in df.columns]
        if missing:
            raise ValueError(f"Missing expected columns: {missing}")
        df = df[feature_names]
    df = df.replace([np.inf, -np.inf], np.nan)
    for col in df.columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].median())
    return df


def score_supervised(flows_df: pd.DataFrame) -> pd.DataFrame:
    X = preprocess_for(flows_df, mlp_feature_names)
    X_scaled = mlp_scaler.transform(X)
    probs = mlp_model.predict(X_scaled, verbose=0)
    pred_idx = np.argmax(probs, axis=1)
    return pd.DataFrame({
        "mlp_label": label_encoder.inverse_transform(pred_idx),
        "mlp_confidence": probs[np.arange(len(probs)), pred_idx],
    }, index=flows_df.index)


def score_unsupervised(flows_df: pd.DataFrame) -> pd.DataFrame:
    X = preprocess_for(flows_df, ae_feature_names)
    X_scaled = ae_scaler.transform(X)
    recon = autoencoder.predict(X_scaled, verbose=0)
    errors = np.mean(np.square(X_scaled - recon), axis=1)
    return pd.DataFrame({
        "ae_error": errors,
        "ae_is_anomaly": errors > ae_threshold,
        "ae_severity_ratio": errors / ae_threshold,  # how far past threshold, for scoring
    }, index=flows_df.index)

print("score_supervised() and score_unsupervised() ready.")


score_supervised() and score_unsupervised() ready.


## Step 4: Fusion — Combine All Three Into One Severity Score

Simple, explainable rule-based fusion (easy to tune and audit — important for a SOC, where analysts need to trust *why* something was flagged):

- **Critical**: Suricata alert AND (supervised attack OR autoencoder anomaly) — a known signature corroborated by behavioral evidence.
- **High**: Suricata alert alone, OR supervised model flags a known attack with high confidence AND autoencoder agrees it's anomalous.
- **Medium**: Only the supervised model fires (confident known-attack classification, no other corroboration), OR Suricata + autoencoder agree but supervised model doesn't (possible attack type the classifier wasn't trained on).
- **Low**: Only the autoencoder fires — anomalous but unconfirmed by any other source. Worth an analyst glance, not an auto-block.
- **None**: Nothing fires — benign.

Tune the confidence/ratio cutoffs below against your own validation data; these are reasonable starting points, not fixed truths.

In [11]:
MLP_CONFIDENCE_CUTOFF = 0.8  # supervised prediction treated as "confident" above this

def fuse_alerts(flows_df: pd.DataFrame, suricata_alerts_df: pd.DataFrame) -> pd.DataFrame:
    result = flows_df.copy().reset_index(drop=True)

    suricata_hit = attach_suricata_flags(flows_df, suricata_alerts_df).reset_index(drop=True)
    mlp_scores = score_supervised(flows_df).reset_index(drop=True)
    ae_scores = score_unsupervised(flows_df).reset_index(drop=True)

    result["suricata_hit"] = suricata_hit
    result = pd.concat([result, mlp_scores, ae_scores], axis=1)

    mlp_flags_attack = (result["mlp_label"] != "BENIGN") & (result["mlp_confidence"] >= MLP_CONFIDENCE_CUTOFF)
    ae_flags_anomaly = result["ae_is_anomaly"]
    suricata_flags = result["suricata_hit"]

    def severity(row_suricata, row_mlp, row_ae):
        if row_suricata and (row_mlp or row_ae):
            return "critical"
        if row_suricata:
            return "high"
        if row_mlp and row_ae:
            return "high"
        if row_mlp:
            return "medium"
        if row_ae:
            return "low"
        return "none"

    result["severity"] = [
        severity(s, m, a) for s, m, a in zip(suricata_flags, mlp_flags_attack, ae_flags_anomaly)
    ]
    return result

print("fuse_alerts() ready.")


fuse_alerts() ready.


## Step 5: Run the Fusion Pipeline

In [12]:
INPUT_CSV = "new_traffic.csv"  # <-- your captured/exported flow data

if os.path.exists(INPUT_CSV):
    new_flows = pd.read_csv(INPUT_CSV)
    fused = fuse_alerts(new_flows, suricata_alerts)

    print(fused["severity"].value_counts())
    print()
    print("Flows above 'low' severity:")
    display(fused[fused["severity"] != "none"].sort_values(
        "severity", key=lambda s: s.map({"critical": 0, "high": 1, "medium": 2, "low": 3, "none": 4})
    ).head(20))
else:
    print(f"'{INPUT_CSV}' not found - set INPUT_CSV to your real flow export to run the full fusion pipeline.")


severity
low     12
none     2
high     1
Name: count, dtype: int64

Flows above 'low' severity:


,src_ip,src_port,dest_ip,dest_port,proto,Label,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,...,Idle Std,Idle Max,Idle Min,suricata_hit,mlp_label,mlp_confidence,ae_error,ae_is_anomaly,ae_severity_ratio,severity
0,192.168.1.105,51322,203.0.113.10,80,TCP,PortScan,9.381051,5.068125,6.065640,6.815033,...,8.876479,9.006637,5.574534,False,PortScan,1.000000,24.895967,True,200.040344,high
1,192.168.1.118,44210,198.51.100.23,445,TCP,Exploit,6.663948,7.470053,5.614289,2.443974,...,4.353080,5.437784,2.927795,False,BENIGN,1.000000,51.969436,True,417.577031,low
2,192.168.1.142,33110,203.0.113.55,53,UDP,BENIGN,0.014489,-1.793649,-0.193856,0.697294,...,1.740346,-0.219495,-0.227253,False,BENIGN,0.999957,0.169584,True,1.362617,low
3,203.0.113.201,443,192.168.1.105,51890,TCP,DDoS,3.919738,6.045556,7.365658,4.519409,...,6.016851,9.359039,4.794979,False,BENIGN,1.000000,4.974516,True,39.970486,low
4,192.168.1.176,60112,198.51.100.88,21,TCP,FTP-Patator,6.659500,5.652353,2.823503,6.515946,...,3.620500,3.077642,8.340923,False,BENIGN,1.000000,26.572100,True,213.508161,low
5,192.168.1.200,55221,203.0.113.99,8080,TCP,Unknown,5.966103,7.079477,7.852604,5.901995,...,4.552268,4.718049,7.363720,False,BENIGN,1.000000,58.361125,True,468.934578,low
6,192.168.1.201,55333,203.0.113.100,8081,TCP,Unknown,9.803856,0.352339,5.379100,6.624108,...,6.298591,7.059789,6.511409,False,BENIGN,1.000000,54.615351,True,438.837093,low
7,192.168.1.202,55444,203.0.113.101,8082,TCP,Unknown,4.112343,3.479620,6.351311,3.632924,...,8.844534,5.429193,9.316850,False,BENIGN,0.999986,4.790389,True,38.491019,low
8,192.168.1.50,40000,203.0.113.150,443,TCP,BENIGN,0.227200,1.124955,-0.225706,0.959171,...,-1.023071,0.993611,0.351797,False,BENIGN,0.999660,0.163097,True,1.310496,low
9,192.168.1.51,40001,203.0.113.151,443,TCP,BENIGN,-0.636626,-3.000316,-0.049125,0.629138,...,-0.537809,-0.509913,1.567866,False,BENIGN,0.999148,0.211124,True,1.696395,low


## Notes

- **Why rule-based fusion instead of another ML layer on top?** A meta-classifier stacked on top of all three signals could squeeze out marginally better numbers, but it turns "why was this flagged critical" into another black box on top of two already-opaque neural nets. For a NIDS a SOC analyst has to trust and act on quickly, an explainable rule table is usually the better trade — you can always revisit this once you have enough labeled fused-alert outcomes to justify a learned meta-model.
- **Retraining cadence**: the autoencoder should be retrained periodically on recent benign traffic — what counts as "normal" drifts over time (new services, new usage patterns), and a stale autoencoder either misses real drift-related attacks or throws increasing false positives.
- **Suricata rule updates**: keep signature sets (e.g. via `suricata-update`) current — the fusion logic is only as good as the weakest-updated layer.
- **Feature/IP retention**: make sure your flow-export pipeline retains the 5-tuple (`src_ip`/`dest_ip`/`src_port`/`dest_port`/`proto`) alongside the numeric features used for ML, purely so Step 2's join can work — it's easy to accidentally drop these during feature selection.
